# Ladder Event Boundary Evaluation

Colab workflow for generating the ladder dataset, running Qwen evaluation, and analyzing results.

In [ ]:
%cd /content

## Clone or update repository

In [ ]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = getpass("GitHub token (leave empty for public repo): ")
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

In [ ]:
%cd /content/vlm-event-boundary
!ls

## Install dependencies

In [ ]:
!pip install -U transformers accelerate qwen-vl-utils decord opencv-python imageio-ffmpeg bitsandbytes

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")

## Generate ladder dataset

In [ ]:
!python scripts/generate_ladder_dataset.py \
  --dataset_version ladder_v1 \
  --samples_per_level 30 \
  --output_root data/ladder_v1 \
  --seed 42

## Check generated annotations

In [ ]:
from pathlib import Path
for path in sorted(Path("data/ladder_v1").glob("level_*/annotations.jsonl")):
    print(path, sum(1 for _ in open(path)))

## Evaluation configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
OUTPUT_DIR = "results"
DATASET_VERSION = "ladder_v1"
EVAL_VIDEO_FPS = 1
EVAL_VIDEO_MAX_PIXELS = 150000
LOAD_IN_4BIT = True

## Smoke test

In [ ]:
!python scripts/run_eval.py \
  --annotation_path data/ladder_v1/level_1_simple/annotations.jsonl \
  --model_name "$MODEL_NAME" \
  --dataset_name ladder_v1_level_1_simple_smoke \
  --output_dir "$OUTPUT_DIR" \
  --max_samples 4 \
  --video_fps "$EVAL_VIDEO_FPS" \
  --video_max_pixels "$EVAL_VIDEO_MAX_PIXELS" \
  --load_in_4bit

## Run all levels

In [ ]:
for level_name in [
    "level_1_simple",
    "level_2_randomized",
    "level_3_static_distractors",
    "level_4_moving_distractors",
]:
    annotation_path = f"data/{DATASET_VERSION}/{level_name}/annotations.jsonl"
    dataset_name = f"{DATASET_VERSION}_{level_name}"
    !python scripts/run_eval.py \
      --annotation_path "$annotation_path" \
      --model_name "$MODEL_NAME" \
      --dataset_name "$dataset_name" \
      --output_dir "$OUTPUT_DIR" \
      --video_fps "$EVAL_VIDEO_FPS" \
      --video_max_pixels "$EVAL_VIDEO_MAX_PIXELS" \
      --load_in_4bit

## Analyze results

In [ ]:
SAFE_MODEL = MODEL_NAME.replace("/", "_")
!python scripts/analyze_results.py \
  --input "results/$SAFE_MODEL" \
  --output_dir "analysis/${DATASET_VERSION}_${SAFE_MODEL}" \
  --plots

In [ ]:
!find results -maxdepth 4 -type f | sort | tail -40
!find analysis -maxdepth 2 -type f | sort